# Testing accuracy of cellprofiler feature extraction for cell size - seeing which method is best

In [ ]:
import os
import numpy as np
import pandas as pd

# plotting
import matplotlib.pyplot as plt

%matplotlib inline
import seaborn as sns
import plotly.express as px
from scipy.stats import shapiro
import re
from scipy import stats
from pathlib import Path
from helpers import *
from plate_preprocessing import *
from mitolyso_plot_functions import *

In [ ]:
## Import your csv files
csvpath = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/CP_Output/postprocessed_csvs/"
stitched_path = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/stitching/segmentation_testset"
filename = "total_combined_cell.csv"
combined_cell_df_mitolyso = pd.read_csv(os.path.join(csvpath, filename))
# filter_df = cell_filters(combined_cell_df_mitolyso)
# display(combined_cell_df_mitolyso.shape)


stitched_csv = "stitched_test_data_v3.csv"
stitched_csv_borders_excluded = "stitched_test_data_v3_borders_excluded.csv"
stitched_nuc_csv = "nuclei_stitched_test_data_v3.csv"
stitched_nuc_csv_borders_excluded = "nuclei_stitched_test_data_v3_borders_excluded.csv"


pre_stitched_cells_df = pd.read_csv(os.path.join(stitched_path, stitched_csv))
pre_stitched_cells_df_borders_excluded = pd.read_csv(
    os.path.join(stitched_path, stitched_csv_borders_excluded)
)

pre_stitched_nuc_df = pd.read_csv(os.path.join(stitched_path, stitched_nuc_csv))
pre_stitched_nuc_df_borders_excluded = pd.read_csv(
    os.path.join(stitched_path, stitched_nuc_csv_borders_excluded)
)


feature_meas = "Cell_AreaShape_Area"

csv_outpath = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/CP_Output/postprocessed_csvs/"
summary_outpath = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/Cell_Size_Data/summary_stats/"

#list out the well - block combos that have poor segmentation and drop them from the datasets
bad_well_ids = {
    #r6
    "r02c03" : ["blockA"],
    "r04c03" : ["blockA,blockB"],#block B is serviceable but poor quality
    "r05c03" : ["blockA,blockB"], 
    "r07c03" : ["blockA,blockB"]
}
wells_to_drop = pd.DataFrame(bad_well_ids)

In [ ]:
combined_cell_df_mitolyso = combined_cell_df_mitolyso[
    combined_cell_df_mitolyso["Staining"].str.startswith("LAMP1-488 + MitoRed")
]
min_x = 0
min_y = 0
# get the max x and y resolutions based on the image dimensions (same for all)
max_x = combined_cell_df_mitolyso["Image_Width_DAPI"][0]  
max_y = combined_cell_df_mitolyso["Image_Height_DAPI"][0]


combined_cell_df_mitolyso_borders_excluded = exclude_borders(
    combined_cell_df_mitolyso, min_x, min_y, max_x, max_y, prefix="Cell_"
)
combined_cell_df_mitolyso_borders_excluded.to_csv(
    os.path.join(csv_outpath, "total_combined_cell_borders_excluded.csv"), index=False
)

# Make the summary stats
combined_cell_df_mitolyso.describe().to_csv(
    os.path.join(summary_outpath, "original_total_combined_cell_stats.csv")
)
combined_cell_df_mitolyso.groupby("AllGroups")["Cell_AreaShape_Area"].describe().to_csv(
    os.path.join(summary_outpath, "original_area_by_passage_group_stats.csv")
)

combined_cell_df_mitolyso_borders_excluded.describe().to_csv(
    os.path.join(summary_outpath, "total_combined_cell_borders_excluded_stats.csv")
)
combined_cell_df_mitolyso_borders_excluded.groupby("AllGroups")[
    "Cell_AreaShape_Area"
].describe().to_csv(
    os.path.join(summary_outpath, "borders_excluded_area_by_passage_group_stats.csv")
)

## Prep the stitched cell dataframe

In [ ]:
# rescale area to the original image by multiplying by the inverse of the rescale factor squared, e.g. for a factor of 1/4 then I can multipy by (4^2)=16
# original_area = measured_area * (1 / rescale_factor**2)


def find_replicate(path):
    replicate_pattern = r"R(\d{1})"  # Matches "RX" where X is the replicate number (placeholder for now)
    match = re.search(replicate_pattern, path)
    if match:
        replicate = int(match.group(1))
    else:
        replicate = None
    return replicate


def find_row_col(well_code):
    rowcol_pattern = r"r(\d{1,2})c(\d{1,2})"  # Matches "RX" where X is the replicate number (placeholder for now)
    match = re.search(rowcol_pattern, well_code)
    if match:
        row_metadata = int(match.group(1))
        col_metadata = int(match.group(2))
    else:
        row_metadata = None
        col_metadata = None
    return row_metadata, col_metadata

#Define another area ratio function
def cell_nuc_area_ratio(row, cell_area_col="Cell_AreaShape_Area", nuc_area_col="Cell_Mean_Nuclei_AreaShape_Area"):
    """
    Calculate the ratio of cell area to nuclear area.
    
    Args:
        row (Series): A row from the DataFrame containing 'Cell_AreaShape_Area' and 'Cell_Mean_Nuclei_AreaShape_Area'.
        
    Returns:
        float: The ratio of cell area to nuclear area.
    """
    if row[nuc_area_col] > 0:
        return row[cell_area_col] / row[nuc_area_col]
    else:
        return np.nan  # Return NaN if nuclear area is zero or negative
    
def prepare_stitched_cells_df_v2(stitched_df, compartment="Cell"):
    stitched_df["Replicate_Number"] = stitched_df.apply(
        lambda row: find_replicate(path=row["Path"]), axis=1
    )
    # stitched_df["Cell_nuclei_area_ratio"] = stitched_df.apply(
    #     lambda row: cell_nuc_area_ratio(row), axis=1
    # )
    stitched_df.rename(columns={"index":"ObjectNumber"})
    try:
        stitched_df.rename(columns={"area":f"{compartment}_AreaShape_Area"})
    except KeyError as e:
        print("Error:" + e + "\n No column found")
    unique_replicates = stitched_df["Replicate_Number"].unique()
    metadata_sliced_dfs = []  # basicalyl split into 3 and rejoin them
    for i, rep in enumerate(unique_replicates):
        if rep == 5:
            map_file = "/mnt/bigdisk1/AllieSpangaro/HTP_fibroblasts_mitolyso/plate_metadata/20250328_rep05_metadata/map.csv"
        elif rep == 6:
            map_file = "/mnt/bigdisk1/AllieSpangaro/HTP_fibroblasts_mitolyso/plate_metadata/20250410_rep06_metadata/map.csv"
        elif rep == 7:
            map_file = "/mnt/bigdisk1/AllieSpangaro/HTP_fibroblasts_mitolyso/plate_metadata/20250501_rep07_metadata/map.csv"
        if os.path.exists(map_file):
            platemap_df = pd.read_csv(map_file)
            platemap_df = platemap_df.drop_duplicates(
                subset=["Metadata_WellRow", "Metadata_WellColumn"]
            )  # drop the dupes here, its ok bcus we only need the passage num
            platemap_df["Metadata_WellRow"] = platemap_df["Metadata_WellRow"].astype(
                int
            )
            platemap_df["Metadata_WellColumn"] = platemap_df[
                "Metadata_WellColumn"
            ].astype(int)

            # join the metadata and add to a list to join
            stitched_cells_df_prefilter = stitched_df.merge(
                platemap_df, on=["Metadata_WellRow", "Metadata_WellColumn"], how="left"
            )
            stitched_cells_df_filter = stitched_cells_df_prefilter[
                stitched_cells_df_prefilter["Replicate_Number"] == rep
            ]
            # display(stitched_cells_df_filter)
            metadata_sliced_dfs.append(stitched_cells_df_filter)
    # display(cell_df[["Metadata_Field","Metadata_WellColumn","Metadata_WellRow","PassageNumber","Drug"]])
    new_stitched_cells_df = pd.concat(metadata_sliced_dfs)
    #display(new_stitched_cells_df.head(30))
    try:
        new_stitched_cells_df["Passage Group"] = new_stitched_cells_df["PassageNumber"].apply(
            passage_group
        )
        new_stitched_cells_df["AllGroups"] = add_drug_to_group(
            new_stitched_cells_df, "Passage Group", "Drug"
        )
    except KeyError:
        new_stitched_cells_df["Passage Group"] = new_stitched_cells_df["PassageNumber_y"].apply(
            passage_group
        )
        new_stitched_cells_df["AllGroups"] = add_drug_to_group(
            new_stitched_cells_df, "Passage Group", "Drug_y"
        )
    except KeyError:
        print("No PassageNumber or Drug columns found in the DataFrame.")
        
    return new_stitched_cells_df
    # display(stitched_cells_df)


def prepare_stitched_cells_df_v1(stitched_cells_df, feature_meas="Cell_AreaShape_Area"):
    stitched_cells_df["Replicate_Number"] = stitched_cells_df.apply(
        lambda row: find_replicate(path=row["Path"]), axis=1
    )
    stitched_cells_df[["Metadata_WellRow", "Metadata_WellColumn"]] = (
        stitched_cells_df.apply(
            lambda row: find_row_col(well_code=row["Well_id"]),
            axis=1,
            result_type="expand",
        )
    )
    # rescale the area by a factor of 16 (inverse of 0.25^2)
    stitched_cells_df["Cell_AreaShape_Area"] = stitched_cells_df.apply(
        lambda x: x["area"] * 16, axis=1
    )
    display(stitched_cells_df)

    unique_replicates = stitched_cells_df["Replicate_Number"].unique()
    metadata_sliced_dfs = []  # basicalyl split into 3 and rejoin them
    for i, rep in enumerate(unique_replicates):
        if rep == 5:
            map_file = "/mnt/bigdisk1/AllieSpangaro/HTP_fibroblasts_mitolyso/plate_metadata/20250328_rep05_metadata/map.csv"
        elif rep == 6:
            map_file = "/mnt/bigdisk1/AllieSpangaro/HTP_fibroblasts_mitolyso/plate_metadata/20250410_rep06_metadata/map.csv"
        elif rep == 7:
            map_file = "/mnt/bigdisk1/AllieSpangaro/HTP_fibroblasts_mitolyso/plate_metadata/20250501_rep07_metadata/map.csv"
        if os.path.exists(map_file):
            platemap_df = pd.read_csv(map_file)
            platemap_df = platemap_df.drop_duplicates(
                subset=["Metadata_WellRow", "Metadata_WellColumn"]
            )  # drop the dupes here, its ok bcus we only need the passage num
            platemap_df["Metadata_WellRow"] = platemap_df["Metadata_WellRow"].astype(
                int
            )
            platemap_df["Metadata_WellColumn"] = platemap_df[
                "Metadata_WellColumn"
            ].astype(int)

            # join the metadata and add to a list to join
            stitched_cells_df_prefilter = stitched_cells_df.merge(
                platemap_df, on=["Metadata_WellRow", "Metadata_WellColumn"], how="left"
            )
            stitched_cells_df_filter = stitched_cells_df_prefilter[
                stitched_cells_df_prefilter["Replicate_Number"] == rep
            ]
            # display(stitched_cells_df_filter)
            metadata_sliced_dfs.append(stitched_cells_df_filter)
    # display(cell_df[["Metadata_Field","Metadata_WellColumn","Metadata_WellRow","PassageNumber","Drug"]])
    stitched_cells_df = pd.concat(metadata_sliced_dfs)
    stitched_cells_df["Passage Group"] = stitched_cells_df["PassageNumber"].apply(
        passage_group
    )
    stitched_cells_df["AllGroups"] = add_drug_to_group(
        stitched_cells_df, "Passage Group", "Drug"
    )
    return stitched_cells_df
    # display(stitched_cells_df)
    
def relate_objects(obj_df_1, obj_df_2, obj1_name = "", obj2_name=""):
    """_summary_

    Args:
        obj_df_1 (_type_): _description_
        obj_df_2 (_type_): _description_
        obj1_name (str, optional): Name of the first object. Defaults to "".
        obj2_name (str, optional): Name of the second object. Defaults to "".

    Returns:
        DataFrame: datframe with the related objects
    """    
    second_objects = []
    for i, row in obj_df_1.iterrows():
        obj_coords = [(obj_df_1["bbox-0"],obj_df_1["bbox-1"]),(obj_df_1["bbox-2"],obj_df_1["bbox-3"])]       
        for j, second_obj in obj_df_2.iterrows():
            if obj_df_2["ImageNumber"] > i:
                break
            # relate if the second object is contained within the parent object
            in_x = (
                second_obj["bbox-0"] <= obj_coords[0][0]
                and second_obj["bbox-1"] >= obj_coords[0][1]
            )
            in_y = (
                second_obj["bbox-2"] <= obj_coords[1][0]
                and second_obj["bbox-3"] >= obj_coords[1][1]
            )
            if in_x and in_y:
                print(f"{obj1_name} contains a {obj2_name} at obj_coords")
                second_objects.append([i, second_obj])
    second_objs_df = pd.DataFrame(second_objects)
    second_obj_means = second_objs_df.groupby("ImageNumber").agg("mean")
    joined_df = obj_df_1.join(second_obj_means, on="ImageNumber", x=obj1_name,y=obj2_name, how="left")
    joined_df["CellNucRatio"] = joined_df.apply(lambda x: x["cell_AreaShape_Area"]/x["nuclei_AreaShape_Area"])
    relate_objects_df = joined_df.dropna()
    return relate_objects_df



In [ ]:
# Set up the dataframes
stitched_cells_df = prepare_stitched_cells_df_v2(pre_stitched_cells_df)
stitched_cells_df_borders_excluded = prepare_stitched_cells_df_v2(
    pre_stitched_cells_df_borders_excluded
)

stitched_nuc_df = prepare_stitched_cells_df_v2(pre_stitched_nuc_df)
stitched_nuc_df_borders_excluded = prepare_stitched_cells_df_v2(pre_stitched_nuc_df_borders_excluded)


In [ ]:
display(stitched_cells_df)
related_df = relate_objects(stitched_cells_df, stitched_nuc_df, "cell", "nuclei")
display(related_df)

group_avg_df = average_groups_by_plate(
    combined_cell_df_mitolyso,
    x_value="AllGroups",
    y_value="Cell_AreaShape_Area",
    replicates="Replicate_Number",
)
group_avg_df_borders_excluded = average_groups_by_plate(
    combined_cell_df_mitolyso_borders_excluded,
    x_value="AllGroups",
    y_value="Cell_AreaShape_Area",
    replicates="Replicate_Number",
)
group_avg_df = apply_shapiro_wilk_test_to_df(group_avg_df, feature_meas)
group_avg_df_borders_excluded = apply_shapiro_wilk_test_to_df(
    group_avg_df_borders_excluded, feature_meas
)
display(group_avg_df)


## code to make the side-by-side comparison plots

In [ ]:
def average_groups_pivot(group_avg_df, x_value, y_value, replicate_col_name):
    """Make a pivot table from the averaged dataframe

    Args:
        group_avg_df (DataFrame): your dataframe output from average_groups_by_plate()
        x_value (string): the grouping variable (x value)
        y_value (string): the quantitavie feature to measure (y value)
        replicate_col_name (string): the variable representing experimental replicates for grouping

    Returns:
        DataFrame: a pivot table
    """
    group_avg_df_pivot = group_avg_df.pivot_table(
        columns=x_value, values=y_value, index=replicate_col_name
    )
    return group_avg_df_pivot


def tukey_statsmodels(data, test_groups, feature):
    """
    Perform a oneway anova test and a pairwise tukey post hoc test using averaged values per replicate
    Returns a dataframe
    """
    from statsmodels.stats.multicomp import pairwise_tukeyhsd

    df = data.copy()
    # groups = getpairs(temp_copy, 'Passage Group')
    # calculate tukey HSD
    tukey = pairwise_tukeyhsd(endog=df[feature], groups=df[test_groups], alpha=0.05)

    # Extract relevant results
    tukey_results = np.array(tukey.summary().data)
    return tukey_results


def pvalues_anova_and_tukeyhsd_posthoc(
    data_df,
    pivot_df,
    x_value,
    y_value,
    replicate_number_col="Replicate_Number",
    desired_pairs=None,
    order=None,
    
):
    """Perform Tukey's HSD post-hoc test on the data.
    See https://github.com/4dcu-be/CodeNuggets/blob/main/Post%20hoc%20tests%20with%20statannotations.ipynb
    Also https://www.biorxiv.org/content/10.1101/2025.02.02.636071v1.full.pdf
    Args:
        data_df (pd.DataFrame): DataFrame table containing the data.
        pivot_df (pd.DataFrame): DataFrame pivot table containing the means.
        x_value (str): Column name for the independent variable.
        y_value (str): Column name for the dependent variable.
        grouping_variable (str): Column name for the replicate number.
        order (list, optional): Order of groups for plotting. Defaults to None.

    Returns:
        pd.DataFrame: DataFrame with Tukey's HSD results.
    """
    from scipy.stats import f_oneway
    from statsmodels.stats.multicomp import pairwise_tukeyhsd

    groups = []  # Convert pivot table to list of groups
    #display(pivot_df)
    for col in pivot_df:
        if col == x_value or col == replicate_number_col:
            print("Skipping col:" + col)
            continue  # Skip the first column (usually the index or grouping variable)
        else:
            print("Adding col:" + col)
            groups.append(pivot_df[col].dropna())
    # One-Way ANOVA
    #display(groups)
    f_value, p_value_anova = f_oneway(
        *list(groups)
    )  # Pass groups as args to run ANOVA on all groups
    print(f"ANOVA F statistic: {f_value}")
    print(f"ANOVA p value: {p_value_anova}")

    # Tukey's HSD (post hoc test)
    if p_value_anova < 0.05:
        tukey_result = pairwise_tukeyhsd(
            endog=data_df[y_value], groups=data_df[x_value], alpha=0.05
        )
        # Extract the data from the Statsmodels SimpleTable
        tukey_data = tukey_result._results_table.data[1:]  # Exclude the header line
        headers = tukey_result._results_table.data[0]  # Get the header line
        tukey_result_df = pd.DataFrame(tukey_data, columns=headers)
        tukey_result_pairs = tukey_result_df[["group1", "group2"]].itertuples(
            index=False, name=None
        )
        pairs = list(tukey_result_pairs)
        p_values = tukey_result_df["p-adj"].tolist()
        display(tukey_result_df)
        return (pairs, p_values)
    else:
        print("ANOVA test is not significant, skipping Tukey's HSD post-hoc test.")
        return ([], [])
    
def anova_with_tukey_posthoc(
    data_df,
    x_value,
    y_value,
    replicate_number_col="Replicate_Number",
    desired_pairs=None,
    order=None,
    display_results=False
):
    from scikit_posthocs import posthoc_tukey

    # Make groups [x,y] for tukey test
    groups = np.unique(data_df[x_value])
    data = []
    for group in groups:
        data.append(data_df[data_df[x_value] == group][y_value])

    anova_result = stats.f_oneway(*data)
    anova_result_pvalue = anova_result.pvalue
    print(f"One-way ANOVA F statistic: {anova_result.statistic}")
    print(f"ANOVA p value: {anova_result_pvalue}")

    if anova_result_pvalue < 0.05:
        # posthoc dunn test
        tukey_df = posthoc_tukey(
            data_df, val_col=y_value, group_col=x_value
        )
        display(tukey_df)

        # melt the dunn_df to long format
        remove = np.tril(np.ones(tukey_df.shape), k=0).astype("bool")
        tukey_df[remove] = np.nan
        molten_df = tukey_df.melt(ignore_index=False).reset_index().dropna()
        
        if display_results:
            display(tukey_df)
            display(molten_df)
            
        dunn_pairs = molten_df[["index", "variable"]].itertuples(index=False, name=None)
        pairs = list(dunn_pairs)
        p_values = molten_df["value"].tolist()
        return (pairs, p_values)

    else:
        print("Oneway ANOVA is not significant, skipping Tukey's post-hoc test.")
        return ([], [])

def kruskal_with_dunn_posthoc(
    data_df,
    x_value,
    y_value,
    replicate_number_col="Replicate_Number",
    desired_pairs=None,
    order=None,
    p_correction="fdr_by",  # graphpad reccomneds two-step Benjamini/Yekutieli method
    display_results=False,
):
    from scikit_posthocs import posthoc_dunn
    # Make groups [x,y] for kruskal test
    groups = np.unique(data_df[x_value])
    data = []
    for group in groups:
        data.append(data_df[data_df[x_value] == group][y_value])

    kruskal_result = stats.kruskal(*data)
    kruskal_pvalue = kruskal_result.pvalue
    print(f"Kruskal-Wallis H statistic: {kruskal_result.statistic}")
    print(f"Kruskal-Wallis p value: {kruskal_pvalue}")

    if kruskal_pvalue < 0.05:
    # posthoc dunn test
        dunn_df = posthoc_dunn(
            data_df, val_col=y_value, group_col=x_value, p_adjust=p_correction
        )
        # melt the dunn_df to long format
        remove = np.tril(np.ones(dunn_df.shape), k=0).astype("bool")
        dunn_df[remove] = np.nan
        molten_df = dunn_df.melt(ignore_index=False).reset_index().dropna()
        
        if display_results:
            display(dunn_df)
            display(molten_df)
        dunn_pairs = molten_df[["index", "variable"]].itertuples(
            index=False, name=None
        )
        pairs = list(dunn_pairs)
        p_values = molten_df["value"].tolist()
        return (pairs, p_values)
        
    else:
        print("Kruskal-Wallis test is not significant, skipping Dunn's post-hoc test.")
        return ([], [])

turkey = anova_with_tukey_posthoc(
    group_avg_df,
    x_value="AllGroups",
    y_value=feature_meas,
    replicate_number_col="Replicate_Number",
    display_results=True,
)
dunn = kruskal_with_dunn_posthoc(
    group_avg_df,
    x_value="AllGroups",
    y_value=feature_meas,
    replicate_number_col="Replicate_Number",
    p_correction='fdr_bh', 
    display_results=True
)
# print(f"Dunn's post-hoc test pairs: {dunn[0]}")
# print(f"Dunn's post-hoc test p-values: {dunn[1]}")

def shapiro_pvalue(
    group_avg_df,
    replicate,
    feature_meas,
    replicate_col_name="Replicate_Number",
    debug=False,
):
    """Function to apply the shapiro wilk test to a dataframe aggregated by replicate for a single feature

    Args:
        group_avg_df (DataFrame): the aggregated dataframe
        replicate (string, int): string or int representation of replicate number
        feature_meas (string): _description_
        replicate_col_name (str, optional): the name of the replicate column. Defaults to "Replicate_Number".
        debug (bool, optional): print out p values. Defaults to False.

    Returns:
        float: p_value from test on that replicate
    """
    from scipy.stats import shapiro

    rep_df = group_avg_df[group_avg_df[replicate_col_name] == replicate]
    # Assume 'df' is your DataFrame and 'feature_meas' is the column to test
    stat, p_value = shapiro(rep_df[feature_meas].dropna())

    if debug:
        print(f"Shapiro-Wilk statistic: {stat}, p-value: {p_value}")
        if p_value < 0.05:
            print("Data is not normally distributed (reject H0)")
        else:
            print("Data is normally distributed (fail to reject H0)")
    return p_value


def apply_shapiro_wilk_test_to_df(
    group_avg_df, feature_meas, replicate_col_name="Replicate_Number", alpha=0.05
):
    """Function to applies the shapiro wilk test row-by-row onto an aggregated dataframe by replicate

    Args:
        group_avg_df (DataFrame): the aggregated dataframe
        feature_meas (string): _description_
        replicate_col_name (str, optional): the name of the replicate column. Defaults to "Replicate_Number".
        alpha (float): the p value threshold. Defaults to p=0.05

    Returns:
        DataFrame: The aggregated dataframe with a "Shaprio_pvalue" column and a boolean "Shapiro_normality" column
    """
    # apply the shapiro-wilk test to a dataframe
    group_avg_df = group_avg_df.dropna()
    group_avg_df["Shapiro_pvalue"] = group_avg_df.apply(
        lambda row: shapiro_pvalue(
            group_avg_df, replicate=row[replicate_col_name], feature_meas=feature_meas
        ),
        axis=1,
    )
    # reject null hypothesis if p < 0.05 - i.e. significant chance that the data is not normally distributed
    group_avg_df["Shapiro_normality"] = group_avg_df.apply(
        lambda row: row["Shapiro_pvalue"] > alpha, axis=1
    )
    return group_avg_df


In [ ]:
def annotate_with_anova_tukey(
    ax,
    pairs,
    data,
    x_value,
    y_value,
    replicate_col_name="Replicate_Name",
    order=None,
    plot="violinplot",
):
    """Add statistical annotations to the plot using one-way ANOVA test.
    see https://statannotations.readthedocs.io/en/latest/custom-test.html for more examples

    Args:
        ax (_type_): _description_
        pairs (_type_): _description_
        group_avg_df (_type_): _description_
        x_value (_type_): _description_
        y_value (_type_): _description_
        order (_type_, optional): _description_. Defaults to None.
        plot (str, optional): _description_. Defaults to "violinplot".

    Returns:
        _type_: _description_
    """
    from statannotations.Annotator import Annotator, StatTest
    from scipy.stats import tukey_hsd

    custom_long_name = "Pairwise Tukey HSD"
    custom_short_name = "tukey"
    custom_func = tukey_hsd
    tukey = StatTest(custom_func, custom_long_name, custom_short_name)
    # tukey = StatTest(tukey_hsd, custom_long_name, custom_short_name)

    # load the custom test
    annotator = Annotator(
        ax, pairs, data=data, order=order, plot=plot
    )  # x=x_value, y=y_value, hue=replicate_col_name,
    annotator.reset_configuration()
    annotator.configure(
        test=tukey,
        text_format="star",  #'simple','full'
        loc="inside",
        hide_non_significant=True,
        color="black",
        verbose=2,
    )
    annotator.apply_and_annotate()
    return ax


def annotate_pairs_with_calculated_pvalues(
    ax,
    data,
    pivot_data,
    x_value,
    y_value,
    replicate_col_name="Replicate_Name",
    test_name="tukey",
    pairs=None,
    order=None,
    plot="violinplot",
):
    """Add statistical annotations to the plot using Tukey's HSD test.
    see https://statannotations.readthedocs.io/en/latest/custom-test.html for more examples

    Args:
        ax (_type_): _description_
        pairs (_type_): _description_
        data (_type_): _description_
        x_value (_type_): _description_
        y_value (_type_): _description_
        replicate_col_name (str, optional): _description_. Defaults to "Replicate_Name".
        test_name (str, optional): _description_. Defaults to "tukey".
        pairs (_type_, optional): _description_. Defaults to None.
        order (_type_, optional): _description_. Defaults to None.
        plot (str, optional): _description_. Defaults to "violinplot".

    Returns:
        _type_: _description_
    """
    from statannotations.Annotator import Annotator

    if pairs is None:
        pairs = getpairs(data, x_value, order=order)

    if test_name in ["kruskal", "dunn","kruskal-wallis"]:
        used_pairs, p_values = kruskal_with_dunn_posthoc(
            data, x_value=x_value, y_value=y_value, order=order, desired_pairs=pairs, p_correction="fdr_bh", display_results=True
        )
    elif test_name in ["anova", "tukey", "tukeyhsd"]:
        # perform anova and tukey's post-hoc test
        used_pairs, p_values = pvalues_anova_and_tukeyhsd_posthoc(
            data, pivot_data, x_value, y_value, order=order, desired_pairs=pairs
        )
    elif test_name == "tukey_v2":
        used_pairs, p_values = anova_with_tukey_posthoc(
            data,
            x_value=x_value,
            y_value=y_value,
            replicate_number_col=replicate_col_name,
            order=order,
            desired_pairs=pairs,
        )
    else:
        raise ValueError(
            f"Test name '{test_name}' is invalid. Use 'tukey', 'anova', 'kruskal', or 'dunn'."
        )
    if used_pairs is None or len(used_pairs) == 0:
        print(f"No significant pairs found for the {test_name} test.")
        return ax
    else:
        annotator = Annotator(
            ax=ax,
            pairs=list(used_pairs),
            data=data,
            plot=plot,
            x=x_value,
            y=y_value,
            order=order,
        )
        annotator.reset_configuration()
        annotator.configure(
            text_format="full",
            test_short_name=test_name,
            pvalue_format_string="{:.3f}",
            #pvalue_format = [[1e-5, "1e-5"], [1e-4, "1e-4"], [1e-3, "0.001"], [1e-2, "0.01"], [5e-2, "0.05"]],
            loc="inside",
            hide_non_significant=True,
            color="black",
            verbose=2,
        )
        annotator.set_pvalues_and_annotate(p_values)
        return ax

def annotate_with_kruskal(
    ax,
    pairs,
    data,
    x_value,
    y_value,
    replicate_col_name="Replicate_Name",
    order=None,
    plot="violinplot",
):
    """Add statistical annotations to the plot using Kruskal-Wallis test.
    see https://statannotations.readthedocs.io/en/latest/custom-test.html for more examples

    Args:
        ax (_type_): _description_
        pairs (_type_): _description_
        data (_type_): _description_
        x_value (_type_): _description_
        y_value (_type_): _description_
        order (_type_, optional): _description_. Defaults to None.
        plot (str, optional): _description_. Defaults to "violinplot".

    Returns:
        _type_: _description_
    """
    from statannotations.Annotator import Annotator

    annotator = Annotator(
        ax, pairs=pairs, data=data, order=order, plot=plot#, x=x_value, y=y_value
    )
    annotator.reset_configuration()
    annotator.configure(
        test="Kruskal",
        text_format="simple",
        # pvalue_format = [[1e-5, "1e-5"], [1e-4, "1e-4"], [1e-3, "0.001"], [1e-2, "0.01"], [5e-2, "0.05"]],
        loc="inside",
        hide_non_significant=True,
        color="black",
        verbose=2,
    )
    annotator.apply_and_annotate()
    return ax


def annotate_legend_with_shapiro(
    ax,
    group_avg_df,
    group_col_name,
    shapiro_col_name="Shapiro_normality",
    palette="pastel",
    title="Replicate",
):
    """add an annotation to the legend of an axis if there is normality via shapiro test

    Args:
        ax (_type_): _description_
        group_avg_df (_type_): _description_
        replicate_col_name (_type_): _description_
    """
    import matplotlib.lines as mlines

    unique_replicates = group_avg_df[group_col_name].unique()
    unique_replicates = sorted(unique_replicates)
    L = plt.legend()
    custom_labels = []
    for rep in unique_replicates:
        label = str(rep)
        shapiro_val = group_avg_df[group_avg_df[group_col_name] == rep][
            shapiro_col_name
        ].iloc[0]
        if shapiro_val:
            label += " (normal)"
        custom_labels.append(label)

    # Create custom legend handles (using the same colors as swarmplot)
    palette = sns.color_palette(palette, n_colors=len(unique_replicates))
    handles = [
        mlines.Line2D(
            [],
            [],
            color=palette[i],
            marker="o",
            linestyle="None",
            markersize=12,
            markeredgecolor="black",
            label=custom_labels[i],
        )
        for i in range(len(unique_replicates))
    ]
    ax.legend_.set_title(title)
    ax.legend(handles=handles, title=title, loc="best")
    return ax


def superviolinplot_helper(
    data_df,
    group_avg_df,
    ax,
    x_value,
    y_value,
    title,
    replicate_col_name,
    pairs=None,
    order=None,
    annotate=False,
    test=None,
):
    if pairs is None:
        pairs = getpairs(data_df, x_value, order=order)
    print(pairs)
    sns.violinplot(
        data=data_df,
        x=x_value,
        y=y_value,  # hue=x_value,
        # palette="Set2",
        split=True,  # using split violin plots - only one side, basically looks like a histogram
        inner="quart",
        color="gainsboro",
        width=0.9,
        linewidth=1.5,
        order=order,
        ax=ax,
    )
    sns.swarmplot(
        data=group_avg_df,
        x=x_value,
        y=y_value,
        hue=replicate_col_name,
        order=order,
        palette="pastel",
        size=12,
        edgecolor="k",
        linewidth=1,
        dodge=False,
        ax=ax,
    )
    # draw a boxplot to show the mean line
    sns.boxplot(
        data=group_avg_df,
        x=x_value,
        y=y_value,
        showmeans=True,
        meanline=True,
        meanprops={"color": "dimgray", "ls": "-", "lw": 2.5},
        medianprops={"visible": False},
        whiskerprops={"visible": False},
        zorder=2,
        showfliers=False,
        showbox=False,
        showcaps=False,
        ax=ax,
    )
    ax.set_title(title)

    # axes[0].text(
    #     x=row[x_value],
    #     y=row[y_value],
    #     s=str(row["Shapiro_normality"]),
    #     color="black",
    #     fontsize=10,
    #     ha="center"
    # )
    # use pivot table to get the average values for each group
    if annotate and test is not None:
        group_avg_pivot_table = average_groups_pivot(
            group_avg_df, x_value, y_value, replicate_col_name
        )
        try:
            ax = annotate_pairs_with_calculated_pvalues(
                ax,
                group_avg_df,
                group_avg_pivot_table,
                x_value,
                y_value,
                replicate_col_name=replicate_col_name,
                test_name=test,
                order=order,
                plot="violinplot",
            )
        except Exception as e:
            print(f"Error annotating with statistical test: {e}")
            # ax = annotate_with_anova_tukey(ax, pairs, group_avg_df_pivot, x_value, y_value, replicate_col_name=replicate_col_name, order=order, plot="violinplot")
        # elif test == "kruskal":
        #     ax = annotate_with_kruskal(
        #         ax,
        #         pairs,
        #         group_avg_pivot_table,
        #         x_value,
        #         y_value,
        #         order=order,
        #         replicate_col_name=replicate_col_name,
        #         plot="violinplot",
        #     )
        ax = annotate_legend_with_shapiro(ax, group_avg_df, replicate_col_name)

    return ax



In [ ]:

def superplot_for_area_threshold_comparisons(
    data_df_1,
    group_avg_df_1,
    data_df_2,
    group_avg_df_2,
    x_value="AllGroups",
    y_value="Cell_AreaShape_Area",
    replicate_col_name="Replicate_Number",
    out_dir="",
    xtitle=None,
    ytitle=None,
    order=None,
    legend=True,
    title1="Original Dataset",
    title2="Excluding Cells Touching Borders",
    annotate=False,
    test=None,
    export_pivot=False,
    show_hist=False,
):
    """Make two side-by-side superplots to compare area between different conditions
    Args:
        data_df_1 (_type_): _description_
        group_avg_df_1 (_type_): _description_
        data_df_2 (_type_): _description_
        group_avg_df_2 (_type_): _description_
        x_value (str, optional): _description_. Defaults to "AllGroups".
        y_value (str, optional): _description_. Defaults to "Cell_AreaShape_Area".
        replicate_col_name (str, optional): _description_. Defaults to "Replicate_Number".
        csv_dir (str, optional): _description_. Defaults to "".
        xtitle (_type_, optional): _description_. Defaults to None.
        ytitle (_type_, optional): _description_. Defaults to None.
    """
    import matplotlib.lines as mlines
    from statannotations.Annotator import Annotator
    from statannotations.stats.StatTest import StatTest

    if order == None:
        order = get_all_group_order()
    pairs = getpairs(data_df_1, x_value, order=order)
    print(pairs)

    if show_hist:
        hist = sns.kdeplot(
            data_df_1, x=y_value, hue=replicate_col_name, palette="pastel"
        )
        plt.show()
        plt.close(hist.figure)
        
        hist2 = px.histogram(
            data_df_1,
            x=y_value,
            color=x_value)
        hist2.show()
        
    fig, axes = plt.subplots(1, 2, figsize=(30, 10), sharey=True, sharex=False)
    # plt.style.use("ggplot")
    sns.set_context("talk", font_scale=1.2)
    sns.set_theme(style="whitegrid")

    # First subplot: Full Dataset
    # sns.stripplot(
    #     data=data_df_1,
    #     x=x_value, y=y_value, hue=x_value,
    #     palette="Set2",
    #     order=order,
    #     ax=axes[0],
    # )
    axes[0] = superviolinplot_helper(
        data_df_1,
        group_avg_df_1,
        axes[0],
        x_value,
        y_value,
        title1,
        replicate_col_name,
        order=order,
        annotate=annotate,
        test=test,
    )
    axes[1] = superviolinplot_helper(
        data_df_2,
        group_avg_df_2,
        axes[1],
        x_value,
        y_value,
        title2,
        replicate_col_name,
        order=order,
        annotate=annotate,
        test=test,
    )

    axes[0].set_title(title1)
    axes[1].set_title(title2)

    if legend:
        axes[0] = annotate_legend_with_shapiro(
            axes[0], group_avg_df_1, replicate_col_name
        )
        axes[1] = annotate_legend_with_shapiro(
            axes[1], group_avg_df_2, replicate_col_name
        )
    else:
        axes[0].legend_.remove()
    if ytitle is not None:
        axes[0].set_ylabel(ytitle)
    if xtitle is not None:
        axes[0].set_xlabel(xtitle)
        axes[1].set_xlabel(xtitle)

    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"combined_cellsize_boxplots_{title2}_{test}.png"))
    plt.show()
    if export_pivot:
        pivot_dir = os.path.join(out_dir, "pivot_tables")
        pivot_dir = Path(pivot_dir)
        Path.mkdir(pivot_dir, exist_ok=True)
        group_avg_df_1_pivot = average_groups_pivot(
            group_avg_df_1, x_value, y_value, replicate_col_name
        )
        group_avg_df_2_pivot = average_groups_pivot(
            group_avg_df_2, x_value, y_value, replicate_col_name
        )
        group_avg_df_1_pivot.to_csv(
            os.path.join(pivot_dir, f"area_pivot_{title1}.csv")
        )  # can plop this into graphpad and see what it tells me
        group_avg_df_2_pivot.to_csv(os.path.join(pivot_dir, f"area_pivot_{title2}.csv"))


superplot_for_area_threshold_comparisons(
    combined_cell_df_mitolyso,
    group_avg_df,
    combined_cell_df_mitolyso_borders_excluded,
    group_avg_df_borders_excluded,
    x_value="AllGroups",
    y_value="Cell_AreaShape_Area",
    out_dir=summary_outpath,
    xtitle="Passage Groups",
    ytitle="Cell Area",
    title1="From Original Dataset",
    title2="Borders Excluded",
    annotate=True,
    test="dunn",
    export_pivot=True,
    show_hist=True,
)


## Make plots and csvs for the comparison btwn regular images and the stitched images

In [ ]:
stitch_group_avg_df = average_groups_by_plate(
    stitched_cells_df,
    x_value="AllGroups",
    y_value="Cell_AreaShape_Area",
    replicates="Replicate_Number",
)
stitch_group_avg_df = apply_shapiro_wilk_test_to_df(
    stitch_group_avg_df, "Cell_AreaShape_Area"
)

# stitch_group_avg_df_borders_excluded = average_groups_by_plate(stitched_cells_df_borders_excluded, x_value='AllGroups', y_value="Cell_AreaShape_Area", replicates='Replicate_Number')
# stitch_group_avg_df_borders_excluded = apply_shapiro_wilk_test_to_df(stitch_group_avg_df_borders_excluded,"Cell_AreaShape_Area")

combined_cell_df_mitolyso["Metadata_Well"] = combined_cell_df_mitolyso.apply(
    lambda x: well_namer(x["Metadata_WellRow"], x["Metadata_WellColumn"]), axis=1
)
combined_cell_df_mitolyso_subset_extrawells = combined_cell_df_mitolyso[
    combined_cell_df_mitolyso["TimepointName"].isin(stitched_cells_df["TimepointName"])
]
combined_cell_df_mitolyso_subset = combined_cell_df_mitolyso_subset_extrawells[
    combined_cell_df_mitolyso_subset_extrawells["Metadata_Well"].isin(
        stitched_cells_df["Metadata_Well_x"]
    )
]

group_avg_df_subset = average_groups_by_plate(
    combined_cell_df_mitolyso_subset,
    x_value="AllGroups",
    y_value="Cell_AreaShape_Area",
    replicates="Replicate_Number",
)
group_avg_df_subset = apply_shapiro_wilk_test_to_df(
    group_avg_df_subset, "Cell_AreaShape_Area"
)

display(group_avg_df_subset)
display(stitch_group_avg_df)

superplot_for_area_threshold_comparisons(
    combined_cell_df_mitolyso_subset,
    group_avg_df_subset,
    stitched_cells_df,
    stitch_group_avg_df,
    out_dir=summary_outpath,
    x_value="AllGroups",
    y_value="Cell_AreaShape_Area",
    order=["P6-10", "P17-19", "P29+", "Doxo"],
    title2="From Stitched Images",
    annotate=True,
    test="kruskal",
    export_pivot=True,
    xtitle="Passage Groups",
    ytitle="Cell Area",
)

stitched_cells_df.to_csv(os.path.join(csv_outpath, "stitched_cells.csv"), index=False)
stitched_cells_df.describe().to_csv(os.path.join(summary_outpath, "stitched_cell_stats.csv"))
stitched_cells_df.groupby("AllGroups")["Cell_AreaShape_Area"].describe().to_csv(
    os.path.join(summary_outpath, "stitched_cell_passage_group_stats.csv")
)


In [ ]:
print(
    pvalues_anova_and_tukeyhsd_posthoc(
        data_df=stitched_cells_df,
        pivot_df=average_groups_pivot(
            stitch_group_avg_df,
            x_value="AllGroups",
            y_value="Cell_AreaShape_Area",
            replicate_col_name="Replicate_Number",
        ),
        x_value="AllGroups",
        y_value="Cell_AreaShape_Area",
        replicate_number_col="Replicate_Number",
        order=None,
    )
)

In [ ]:
#Compare stitched cells to stitched cells with borders excluded
display(stitched_cells_df_borders_excluded.shape, stitched_cells_df.shape) #make sure they aren't the same size to validate
stitch_group_avg_df_borders_excluded = average_groups_by_plate(
    stitched_cells_df_borders_excluded,
    x_value="AllGroups",
    y_value="Cell_AreaShape_Area",
    replicates="Replicate_Number",
)
stitch_group_avg_df_borders_excluded = apply_shapiro_wilk_test_to_df(
    stitch_group_avg_df_borders_excluded, "Cell_AreaShape_Area"
)

superplot_for_area_threshold_comparisons(
    stitched_cells_df,
    stitch_group_avg_df,
    stitched_cells_df_borders_excluded,
    stitch_group_avg_df_borders_excluded,
    order=["P6-10", "P17-19", "P29+", "Doxo"],
    title1="From Stitched Images",
    title2="Borders Excluded",
    out_dir=summary_outpath,
    x_value="AllGroups",
    y_value="Cell_AreaShape_Area",
    xtitle="Passage Groups",
    annotate=True,
    test="kruskal",
    export_pivot=True,
    #show_hist=True,
)

stitched_cells_df_borders_excluded.to_csv(
    os.path.join(csv_outpath, "stitched_cells_borders_excluded.csv"), index=False
)
stitched_cells_df_borders_excluded.describe().to_csv(
    os.path.join(summary_outpath, "stitched_cells_borders_excluded_stats.csv")
)
stitched_cells_df_borders_excluded.groupby("AllGroups")[
    "Cell_AreaShape_Area"
].describe().to_csv(
    os.path.join(summary_outpath, "stitched_cell_borders_excluded_passage_group_stats.csv")
)
